# 🧪 Compatibility Test Suite — Colab GPU Runner\n\nThis notebook runs the full compatibility test suite on a Colab GPU.\n\n**Requirements:** T4 (16 GB) or better. A100 recommended for 4D models.\n\n---

In [ ]:
# ── Cell 1: Verify GPU & Clone Repo ──────────────────────────────────────────
import subprocess, os

# Check GPU
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(f"🖥️  GPU: {gpu_info.stdout.strip()}")

# Clone repo
if not os.path.exists("/content/Research"):
    subprocess.run(["git", "clone", "https://github.com/Snehpatel101/Research.git", "/content/Research"], check=True)
    print("✅ Repo cloned")
else:
    subprocess.run(["git", "-C", "/content/Research", "pull", "--ff-only"], check=True)
    print("✅ Repo updated")

os.chdir("/content/Research")
print(f"📁 Working dir: {os.getcwd()}")

In [ ]:
# ── Cell 2: Install Dependencies ─────────────────────────────────────────────
import subprocess

subprocess.run(["pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

# Verify key imports
import torch
print(f"✅ PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM:   {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ── Cell 3: Run Unit Tests (quick sanity check) ─────────────────────────────
import subprocess, os
os.chdir("/content/Research")
os.environ["TORCHDYNAMO_DISABLE"] = "1"

result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-x", "-q", "--tb=short"],
    capture_output=True, text=True, timeout=300
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("⚠️ STDERR:", result.stderr[-2000:])
print(f"\n{'✅' if result.returncode == 0 else '❌'} Unit tests exit code: {result.returncode}")

In [ ]:
# ── Cell 4: Run Compatibility Tests (--skip-4d, 14 tests) ───────────────────
import subprocess, os, time
os.chdir("/content/Research")
os.environ["TORCHDYNAMO_DISABLE"] = "1"

print("🚀 Starting compatibility test suite (--skip-4d)...")
print("   This tests all 2D + 3D model combinations (14 tests)")
print("   Expected time: ~15-25 min on T4\n")

start = time.time()
result = subprocess.run(
    ["python", "scripts/compatibility_test.py", "--skip-4d"],
    capture_output=True, text=True, timeout=3600
)
elapsed = time.time() - start

print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)
if result.returncode != 0:
    print("⚠️ STDERR:", result.stderr[-3000:])
print(f"\n⏱️  Total time: {elapsed/60:.1f} min")
print(f"{'✅' if result.returncode == 0 else '❌'} Compatibility tests exit code: {result.returncode}")

## ⬇️ Optional: Full Suite with 4D Models (A100 recommended)\nOnly run the cell below if you have an A100 (40 GB+). PatchTST and iTransformer need significant VRAM.

In [ ]:
# ── Cell 5: Run FULL Compatibility Tests (all 17 tests, A100 only) ──────────
import subprocess, os, time
os.chdir("/content/Research")
os.environ["TORCHDYNAMO_DISABLE"] = "1"

print("🚀 Starting FULL compatibility test suite (all 17 tests incl. 4D)...")
print("   ⚠️  Requires A100 (40 GB+) for PatchTST / iTransformer")
print("   Expected time: ~30-45 min on A100\n")

start = time.time()
result = subprocess.run(
    ["python", "scripts/compatibility_test.py"],
    capture_output=True, text=True, timeout=7200
)
elapsed = time.time() - start

print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)
if result.returncode != 0:
    print("⚠️ STDERR:", result.stderr[-3000:])
print(f"\n⏱️  Total time: {elapsed/60:.1f} min")
print(f"{'✅' if result.returncode == 0 else '❌'} FULL test suite exit code: {result.returncode}")